# Siddhidatri: one calibrated verdict from three detectors

Per-minute scores from a learned detector (trained without the minute's machine group) and two statistical ones. Fuse them with logistic regression, held out by group, and check whether fusion beats the best single detector and whether its probabilities mean what they say.

Siddhidatri is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import pandas as pd, numpy as np
d = pd.read_parquet(find("detector_scores.parquet")).fillna(0)
S = ["shailaputri", "max_robust_z", "pca"]
print(d.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
rows = []
for held in sorted(d["group"].unique()):
    tr, te = d[d["group"] != held], d[d["group"] == held]
    fused = LogisticRegression(max_iter=1000).fit(tr[S], tr["anomaly"]).predict_proba(te[S])[:, 1]
    row = {"held out": held, "fused AUC": roc_auc_score(te["anomaly"], fused),
           "fused Brier": brier_score_loss(te["anomaly"], fused)}
    for s in S:
        row[f"{s} AUC"] = roc_auc_score(te["anomaly"], te[s])
    rows.append(row)
pd.DataFrame(rows).round(3)

**Reading it:** if fused AUC barely beats the learned detector alone, fusion buys little. Siddhidatri matched Shailaputri's F1 (0.534 against 0.530) with far more false alarms (47.9 against 4.0 per 1000). A fusion that catches more at the same false-alarm rate would be a new result.